In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

file_path = os.path.expanduser("~/Desktop/regge.xlsx")
df = pd.read_excel(file_path)

model_trust = df[
    ["Preis", "Seller Rating", "Seller Type", "zahl der verkauften artikel"]
].copy()

def clean_price(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("EUR", "").replace("€", "").replace(".", "").replace(",", ".").strip()
    try:
        return float(x)
    except:
        return np.nan

def clean_rating(x):
    if pd.isna(x) or x == "N/A":
        return np.nan
    x = str(x).replace("%", "").replace(",", ".").strip()
    try:
        return float(x)
    except:
        return np.nan

def clean_number(x):
    if pd.isna(x) or x == "N/A":
        return np.nan
    x = str(x).replace(".", "").replace(",", ".").replace("+", "").strip()
    try:
        return float(x)
    except:
        return np.nan

model_trust["price"] = model_trust["Preis"].apply(clean_price)
model_trust["seller_rating"] = model_trust["Seller Rating"].apply(clean_rating)
model_trust["seller_sales_count"] = model_trust["zahl der verkauften artikel"].apply(clean_number)

model_trust = model_trust.drop(columns=["Preis", "Seller Rating", "zahl der verkauften artikel"])
model_trust = model_trust.dropna()

model_encoded = pd.get_dummies(
    model_trust,
    columns=["Seller Type"],
    drop_first=True
)

X = model_encoded.drop("price", axis=1)
y = model_encoded["price"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

trust_model = sm.OLS(y, X).fit()

trust_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.176
Model:                            OLS   Adj. R-squared:                  0.163
Method:                 Least Squares   F-statistic:                     13.78
Date:                Fri, 08 May 2026   Prob (F-statistic):           3.48e-08
Time:                        13:22:44   Log-Likelihood:                -1152.0
No. Observations:                 198   AIC:                             2312.
Df Residuals:                     194   BIC:                             2325.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                801.0963     12.141     65.983      0.000     777.151     825.041
seller_rating          0.4662      0.142      3.274      0.001       0.185       0.747
seller_sales_count  2.802e-05   1.05e-05      2.678      0.008    7.39e-06    4.87e-05
Seller Type_Privat   -22.3125     14.486     -1.540      0.125     -50.882       6.257
==============================================================================
Omnibus:                       28.279   Durbin-Watson:                   1.531
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               38.679
Skew:                           0.868   Prob(JB):                     3.99e-09
Kurtosis:                       4.293   Cond. No.                     1.82e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.82e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [2]:
print("Observations:", len(y))
print("R²:", trust_model.rsquared)
print("Adjusted R²:", trust_model.rsquared_adj)

Observations: 198
R²: 0.1756758415864006
Adjusted R²: 0.16292856078619034
